# L2-05 配套 notebook：消融与证伪设计

对应课文 [`docs/lessons/L2-05-实验方法-消融与证伪设计.md`](../docs/lessons/L2-05-实验方法-消融与证伪设计.md)。

本 notebook 把课文的三类自欺来源与三个判据阈值做成**可以跑、可以看**的东西。
全部用**纯 NumPy 合成数据**，**不加载任何真实评分**，因此这里的数字演示的是
**机制**，不是本赛的任何量。

**它回答六个问题：**

| 单元 | 问题 |
|---|---|
| 1 | 换随机种子再比较，为什么什么都证明不了？种子噪声有多大？ |
| 2 | 扫 $k$ 个候选取最大值，虚高多少？对表课文 §2.3 |
| 3 | 怎么把地板套到一个具体改动上？三种情形分别会输出什么？ |
| 4 | 重抽真值侧 vs 重抽模型侧，为什么必须是前者？ |
| 5 | 「总分涨、DE 三项跌」是什么样子？只看总分漏掉了什么？ |
| 6 | 消融顺序为什么影响总成本？ |

**前置阅读**：[L1-02 证据基线与噪声地板](../docs/lessons/L1-02-证据基线与噪声地板.md) §5.4（地板数字）
与 §8.1（判定流程）、[L1-01 领域地图与五种赌注](../docs/lessons/L1-01-领域地图与五种赌注.md) §3.3（消融顺序初版）。

## 单元 0｜环境与记号

统一记号，后面所有单元都用它。

- $\Delta$：观测到的分数差（改动后 − 改动前）
- $T$：地板阈值（判定差异够不够大）
- $\sigma_{\text{model}}$：模型侧随机性带来的标准差
- $\sigma_{\text{truth}}$：真值侧抽样带来的标准差

In [ ]:
import numpy as np

# 固定种子，保证本 notebook 可重复。
rng = np.random.default_rng(20260918)

print("NumPy:", np.__version__)
print()
print("记号约定：")
print("  Δ            = 观测分数差（改动后 - 改动前）")
print("  T            = 地板阈值")
print("  sigma_model  = 模型侧随机性标准差（换种子会变）")
print("  sigma_truth  = 真值侧抽样标准差（换真值细胞会变）")
print()
print("本 notebook 全部用合成数据演示机制，不代表本赛的任何量。")

## 单元 1｜种子噪声：为什么「换种子再比较」什么都不证明

课文 §2.1 的场景：用**种子 1** 跑基线、**种子 2** 跑改动，看到 $\Delta = 0.007$，
就以为这是改动的效果。

单元 1 构造一个**真实效果恰好为 0** 的改动，看只用两个不同种子比较时，
你会「看到」多大的差异。

In [ ]:
# 构造：某个改动真实效果 = 0。每次测量 = 真实值 + 种子噪声。
TRUE_EFFECT = 0.0
SIGMA_MODEL = 0.006          # 模型侧标准差（合成值）

N_TRIALS = 20000             # 为了看分布，做很多次"两两比较"

# 每次比较：种子A 的观测 vs 种子B 的观测，两者真实值相同
a = rng.normal(TRUE_EFFECT, SIGMA_MODEL, N_TRIALS)
b = rng.normal(TRUE_EFFECT, SIGMA_MODEL, N_TRIALS)
obs_delta = a - b

print("真实效果 = %.4f（这个改动其实什么都没做）" % TRUE_EFFECT)
print("模型侧 sigma = %.4f" % SIGMA_MODEL)
print()
print("只用两个不同种子比较，会观测到：")
print("  Δ 的标准差        = %.4f" % obs_delta.std())
print("  |Δ| > 0.004 的比例 = %.1f%%" % (100 * (np.abs(obs_delta) > 0.004).mean()))
print("  |Δ| > 0.008 的比例 = %.1f%%" % (100 * (np.abs(obs_delta) > 0.008).mean()))
print("  |Δ| 的最大值       = %.4f" % np.abs(obs_delta).max())
print()
print("解读：即使真实效果是 0，两两比较也经常给出 0.004-0.008 的'改进'。")
print("      课文 §2.1 里 0.0071 的那个数字，完全落在这个虚假范围内。")
print()
print("结论：比较两个版本时，种子必须固定；否则你测的是种子噪声，不是改动效果。")

## 单元 2｜多重比较：扫 $k$ 个候选取最大值会虚高多少

课文 §2.3 的表。这里把它算出来。

**设定**：所有 $k$ 个候选的真实效果**完全相同**（都是 0），每次测量含标准差
$\sigma$ 的噪声。取这 $k$ 个观测的最大值。即使真实效果为 0，
**最大值也会系统性地为正**。

In [ ]:
SIGMA = 1.0
N_REP = 200000

print("真实效果全部为 0，取 k 个观测的最大值：")
print()
print("%6s %12s %12s" % ("k", "最大值期望(σ)", "最大值>0比例"))
print("-" * 34)
for k in [1, 2, 3, 6, 12, 30, 60]:
    draws = rng.normal(0.0, SIGMA, size=(N_REP, k))
    maxima = draws.max(axis=1)
    print("%6d %12.3f %11.1f%%" % (k, maxima.mean(), 100 * (maxima > 0).mean()))

print()
print("提示：k=6 时约 +1.27σ，与课文 §2.3 的表一致。")
print()
print("换算到分数：若 σ 对应 0.005 分，则仅仅因为扫了 6 个点，")
print("            就可能看到约 0.006 的假增益 —— 恰好是课文里那个 0.0071。")
print()
print("推论（课本 §6.2 判据二）：扫描取最优的数字，必须在独立确认集上复测一次。")

## 单元 3｜把地板套到一个具体改动上

课文 §6.1 与 §6.4 的判据一：$|\Delta| > T$ 才进入下一步。

单元 3 实现一个机械判定函数，并把**三种情形**各跑一次：

| 情形 | Δ | T | 应该输出 |
|---|---:|---:|---|
| A | 0.0020 | 0.0050 | 不判定（可能只是功效不足） |
| B | 0.0071 | 0.0050 | 进入符号稳定性检验 |
| C | −0.0080 | 0.0050 | 进入符号稳定性检验（方向是负的） |

In [ ]:
def judge_by_floor(delta, floor, floor_scale="unknown"):
    """课文 §6.4 判据一。返回 (是否跨过, 说明)。"""
    if floor_scale != "avg_score":
        return None, ("地板尺度不是 avg_score。课文 §6.1 明确："
                      "L1-02 的 28.8/29.7/32.9 不能直接搬到这里。")
    if abs(delta) <= floor:
        return False, "|Δ| <= T：不判定为改进。注意这不等于'证明无改进'。"
    return True, "|Δ| > T：进入判据三（真值侧重抽看符号稳定性）。"


CASES = [
    ("A 改进小于地板", 0.0020),
    ("B 改进大于地板", 0.0071),
    ("C 方向为负",    -0.0080),
]
T_EXAMPLE = 0.0050

print("地板阈值 T = %.4f（示意值；真实 T 须由评估回路在 avg_score 尺度上产出）\n" % T_EXAMPLE)
for name, d in CASES:
    ok, why = judge_by_floor(d, T_EXAMPLE, floor_scale="avg_score")
    tag = "跨过" if ok else "未跨过"
    print("%-18s Δ = %+.4f  -> %-6s  %s" % (name, d, tag, why))

print()
print("--- 先演示误用：把 L1-02 的群体距离地板当 avg_score 地板 ---")
ok, why = judge_by_floor(0.0071, 28.8, floor_scale="vector_distance")
print("  Δ=0.0071 与 T=28.8 ->", why)
print()
print("这正是课文中反复警告的那个错：地板是分尺度的，不能跨尺度复用。")

## 单元 4｜真值侧重抽 vs 模型侧重抽（本 notebook 的核心）

课文 §6.3 的操作要求：**固定模型、只重抽真值侧**。

单元 4 构造一个**真实效果为 0.0054** 的改动，然后用两种方式重抽 $N=20$ 次：

- **方式 A（错误）**：换模型种子重跑——只动 $\sigma_{\text{model}}$
- **方式 B（正确）**：固定模型，只重抽真值细胞——只动 $\sigma_{\text{truth}}$

如果 $\sigma_{\text{truth}} \gg \sigma_{\text{model}}$，两种方式会给出**完全不同**的结论。

In [ ]:
TRUE_DELTA = 0.0054          # 改动的真实效果（课文 §8.3 那个数字）
SIGMA_MODEL = 0.0006         # 课文 §8.3 里他们测到的"0.0006"
SIGMA_TRUTH = 0.0040         # 真值侧标准差（他们没测；这里假设远大于模型侧）
N_REP = 20

# 方式 A：换模型种子。真值固定，只有模型噪声在动。
model_side = rng.normal(TRUE_DELTA, SIGMA_MODEL, N_REP)

# 方式 B：固定模型，只重抽真值侧。
truth_side = rng.normal(TRUE_DELTA, SIGMA_TRUTH, N_REP)

def report(name, samples):
    lo, hi = np.percentile(samples, [0.5, 99.5])
    crosses = lo <= 0 <= hi
    print("%-26s 均值 %+.4f  标准差 %.4f" % (name, samples.mean(), samples.std()))
    print("%-26s 99%% 区间 [%+.4f, %+.4f]  跨 0: %s"
          % ("", lo, hi, "是" if crosses else "否"))
    return crosses

print("真实效果 = %+.4f\n" % TRUE_DELTA)
cross_a = report("A 换模型种子（错误）", model_side)
print()
cross_b = report("B 重抽真值侧（正确）", truth_side)
print()
print("解读：")
print("  方式 A 的 99% 区间几乎贴着 +0.0054，看起来'非常稳定' ——")
print("  这正是课文 §8.3 里那个团队宣布'改进确认'的依据。")
print("  但方式 B 的区间宽得多，%s。" % ("跨越 0，因此不能判定" if cross_b else "仍不跨 0"))
print()
print("关键：地板 T 的定义（L1-02 §3.3）就是真值侧的抽样方差。")
print("      用模型侧标准差去比 Δ，比较对象根本不是同一个量。")
print("      如果 sigma_truth 远大于 sigma_model，20 次模型重跑的信息量")
print("      近似于 20 次测量同一个量 —— 算力几乎全部浪费。")

## 单元 5｜「总分涨、DE 三项跌」长什么样

课文 §4.2 的 #11，也是 §8.2 诊断题的机制。

单元 5 构造一个改动：把生成阶段的**细胞间变异按系数 $c$ 缩小**。
机制是：三项 DE 指标建立在 Wilcoxon 检验上，检验需要组内变异；
缩小组内变异会让检验更容易显著，从而**人为扩大预测显著基因集合**。

In [ ]:
# 构造：c 越小 -> 总分越高（因为 DE 分母被污染），但 DE 指标本身在跌。
C_VALUES = [0.5, 0.7, 0.9, 1.0]

# 合成数据：形状取自课文 §8.2 诊断题的表。
AVG   = {0.5: -0.0331, 0.7: -0.0340, 0.9: -0.0372, 1.0: -0.0398}
DIRFID = {0.5: 0.182, 0.7: 0.238, 0.9: 0.311, 1.0: 0.349}
JACC  = {0.5: 0.041, 0.7: 0.063, 0.9: 0.089, 1.0: 0.112}

print("%5s %12s %14s %16s" % ("c", "avg_score", "方向保真度", "显著基因 Jaccard"))
print("-" * 52)
for c in C_VALUES:
    print("%5.1f %12.4f %14.3f %16.3f" % (c, AVG[c], DIRFID[c], JACC[c]))

base = AVG[1.0]
best = AVG[0.5]
print()
print("总分：c=1.0 -> c=0.5 上涨 %.4f（看起来是显著改进）" % (best - base))
print("方向保真度：%.3f -> %.3f，跌了 %.0f%%"
      % (DIRFID[1.0], DIRFID[0.5], 100 * (1 - DIRFID[0.5] / DIRFID[1.0])))
print("显著基因 Jaccard：%.3f -> %.3f，跌了 %.0f%%"
      % (JACC[1.0], JACC[0.5], 100 * (1 - JACC[0.5] / JACC[1.0])))

print()
print("--- 单调性检查：如果 c 变小真的更好，那 c -> 0 应该最好 ---")
print("    c = 1.0  -> avg_score = %+.4f" % AVG[1.0])
print("    c = 0.5  -> avg_score = %+.4f" % AVG[0.5])
print("    c = 0.0  -> avg_score = -0.8100   （L1-02 §10 记录的极端情况，参赛者自报）")
print()
print("单调上升的序列末端是悬崖，而不是平台 —— 这是'评分被操纵'")
print("而不是'模型变好'的典型签名。")
print()
print("判据（课文 §4.2 第二条硬规则）：任何只报告总分的消融都不算完成。")
print("      至少同时报告 (a) 逐指标成绩单，(b) 逐留出背景的成绩。")

## 单元 6｜消融顺序为什么影响总成本

课文 §3.1：把昂贵实验排在便宜实验前面，是最容易犯的流程错误。

单元 6 用一个简单的成本模型说明：假设每一步有成本 $c_i$ 和「提前终止的概率」
$p_i$（即这一步就能否掉后续所有昂贵步骤）。比较不同顺序的**期望总成本**。

In [ ]:
# 每一步：(名称, 成本(任意单位), 终止概率)
# 终止概率 = 这一步失败从而不需要继续做后面的概率。
STEPS = [
    ("0 无变化基准",      0.01, 0.02),
    ("1 均值基线 B0",     0.05, 0.10),
    ("2 加性/低秩 B1/B3", 0.30, 0.25),
    ("3 相似度加权 B2",   0.40, 0.15),
    ("4 接口修正",        0.20, 0.30),
    ("5 计数生成器",      0.50, 0.20),
    ("6 轻量适配",        5.00, 0.40),
    ("7 微调/增容量",    40.00, 0.00),
]


def expected_cost(order):
    """按顺序执行；一旦某步终止就不再继续。返回期望总成本。"""
    total = 0.0
    survive = 1.0                      # 走到这一步的概率
    for idx in order:
        name, cost, stop = STEPS[idx]
        total += survive * cost
        survive *= (1 - stop)
    return total


cost_natural = expected_cost(range(len(STEPS)))                       # 按课文顺序
cost_reversed = expected_cost(range(len(STEPS) - 1, -1, -1))          # 昂贵在前

print("消融顺序的成本模型（成本单位为任意单位，仅用于比较）\n")
print("%-22s %10s %10s" % ("步骤", "成本", "终止概率"))
print("-" * 44)
for name, cost, stop in STEPS:
    print("%-22s %10.2f %9.0f%%" % (name, cost, 100 * stop))

print()
print("按课文 §3.2 的顺序（便宜在前）：期望成本 = %8.2f" % cost_natural)
print("反过来（昂贵在前）            ：期望成本 = %8.2f" % cost_reversed)
print("倍数                        ：%.2fx" % (cost_reversed / cost_natural))

print()
print("解读：")
print("  便宜步骤的终止概率把昂贵步骤'提前砍掉'，这正是排序的价值。")
print("  课文 §3.3 原则一：先排除'测量错了'，再排除'模型不行'。")
print("  如果输出合同的基因顺序就错了，微调一周也不会得到可解释的结果。")
print()
print("注意：这里的成本与概率都是示意值，真实数字要在租卡后回填（课文 §7.3）。")

## 单元 7｜证伪清单自检

课文 §4.1 给了 13 条证伪条件。单元 7 把它编成表，并**机械检查每一行是否都有
可观测的失败信号**——这是课文 §4.3 的要求：写不出可观测的证伪条件，就不要开始。

In [ ]:
# 课文 §4.1 的 13 条。每行：(改动, 是否可观测, 观测手段)
CHECKLIST = [
    (1,  "换损失函数（Energy -> MSE）",        True,  "逐指标成绩单：分布指标 vs 幅度指标"),
    (2,  "去掉残差加回",                       True,  "预测与 NTC 的逐基因相关"),
    (3,  "Stack 上下文放更多示例细胞",          True,  "显存占用与 batch size 记录"),
    (4,  "上下文放同类细胞而非随机细胞",        True,  "同背景 vs LOCO 双协议对照"),
    (5,  "scGPT 嵌入替换随机靶点嵌入",          True,  "该对照臂 + 同种子多重复 vs 地板"),
    (6,  "离散 perturbation ID 表示靶点",       True,  "零覆盖靶点子集上的分数"),
    (7,  "加辅助损失项",                       True,  "固定划分重跑 + 换划分对照"),
    (8,  "聚合统计量 mean -> median",           True,  "逐背景分解（差异是否集中在有异常值的背景）"),
    (9,  "加相似背景加权 B2",                   True,  "逐背景分解，特别看跨实验室/跨平台那格"),
    (10, "B3 低秩交互用 ID embedding",          True,  "零覆盖靶点子集"),
    (11, "缩小细胞间方差",                      True,  "逐指标成绩单：总分涨、DE 三项跌"),
    (12, "集成多个模型",                       True,  "多 seed + 地板比较（增益常 ≤ 地板）"),
    (13, "扫描超参取最优值",                    True,  "独立确认集上复测"),
]

print("%4s %-34s %-8s %s" % ("#", "改动", "可观测", "观测手段"))
print("-" * 100)
for n, change, obs, how in CHECKLIST:
    print("%4d %-34s %-8s %s" % (n, change, "✓" if obs else "✗", how))

n_unobservable = sum(1 for _, _, obs, _ in CHECKLIST if not obs)
print()
print("总条数：%d    无观测手段的：%d" % (len(CHECKLIST), n_unobservable))

# 三条「总分上看不出来」的，单独列出来
HIDDEN = [1, 9, 11]
print()
print("其中总分上【看不出来】的 %d 条，必须靠逐指标/逐背景分解：" % len(HIDDEN))
for n in HIDDEN:
    for m, change, _, how in CHECKLIST:
        if m == n:
            print("  #%-3d %-30s -> %s" % (m, change, how))

print()
print("自检标准（课文 §7.2）：把表交给一个没参与的人，")
print("他能否只根据'观测手段'列判断一次实验是成功还是失败。")
print("只要有一行让他需要来问你，那一行就没写清楚。")

## 单元 8｜自检清单与三件没做的事

读完本 notebook，你应该能不看课文回答：

1. 为什么「换种子再比较」得到的 $\Delta$ 不能作为改动效果的证据？（单元 1）
2. 扫 6 个候选取最大值，会带来多少 $\sigma$ 的虚高？为什么扫描取最优的数字必须上确认集？（单元 2）
3. 判据一输出「未跨过」时，正确的表述是什么？（单元 3）
4. 为什么必须重抽**真值侧**而不是模型侧？（单元 4）
5. 「总分涨、DE 三项跌」为什么是评分被操纵而不是模型变好？（单元 5）
6. 三类自欺里，哪一类不需要任何错误、只看公式就会发生？（§2.3）

In [ ]:
print("=" * 74)
print("自检：本 notebook 的结论与课文对照")
print("=" * 74)
checks = [
    ("种子噪声", "真实效果为 0 时两两比较也会给出 0.004-0.008 的假改进", "课文 §2.1 / 单元 1"),
    ("多重比较", "k=6 带来约 +1.27σ 虚高；扫描数字须上确认集", "课文 §2.3 / §6.2 / 单元 2"),
    ("判据一", "|Δ| <= T -> '现有证据不足以判定'，不是'证明无改进'", "课文 §6.1 / 单元 3"),
    ("判据三", "地板定义在真值侧；重抽真值侧才测到 T 对应的量", "课文 §6.3 / 单元 4"),
    ("逐指标分解", "总分涨、DE 三项跌 = 评分口径被操纵", "课文 §4.2 / 单元 5"),
    ("消融顺序", "便宜步骤的终止概率把昂贵步骤提前砍掉", "课文 §3.2 / 单元 6"),
]
for q, a, ref in checks:
    print("  • %-14s -> %-52s [%s]" % (q, a, ref))

print()
print("三件本 notebook 没做的事：")
print("  1. 没有加载任何真实评分 —— 全部合成数据，演示的是机制不是本赛的量。")
print("  2. 没有给出 avg_score 尺度的地板 T —— 那必须由 L3-01 的评估回路产出。")
print("  3. 没有执行任何真实消融 —— 课文的 13 条是待执行清单，不是已完成结果。")

print()
print("=" * 74)
print("ALL CELLS OK")
print("=" * 74)